# September Week 1 (Part A) — Brownian Motion

**Project:** AI/ML-Augmented Black–Scholes Framework
**Plan reference:** §15.4.1 (learning objectives), §15.4.2 (computational work)
**Text:** Carol Alexander, *Market Risk Analysis Vol. I*, §I.3.7

---

### Learning objective (§15.4.1)

The Brownian-motion increment satisfies

$$\Delta W = W_{t+\Delta t} - W_t, \qquad \Delta W \sim N(0, \Delta t)$$

or equivalently

$$\Delta W = \sqrt{\Delta t}\,Z, \qquad Z \sim N(0,1)$$

The whole of Brownian motion follows from this one statement plus independence of
non-overlapping increments.

### Computational work (§15.4.2)

| # | Task | Section |
|---|---|---|
| 1 | Generate standard normal random variables | 1 |
| 2 | Generate Brownian increments | 2 |
| 3 | Construct complete Brownian-motion paths | 3 |
| 4 | Verify numerically that $E[\Delta W] \approx 0$ | 4 |
| 5 | Verify $\mathrm{Var}(\Delta W) \approx \Delta t$ | 4 |
| 6 | Repeat the experiment for different values of $\Delta t$ | 4 |
| 7 | Demonstrate numerically that $\mathrm{SD}(\Delta W) \propto \sqrt{\Delta t}$ | 4 |

### Additional experiments (not required by §15.4.2)

| # | Task | Why |
|---|---|---|
| 8 | Independence and stationarity of increments | These are defining properties of Brownian motion; without them the increments are merely normal noise |
| 9 | Quadratic variation | $\sum(\Delta W)^2 \to T$ is the mechanism that produces the $\tfrac12\sigma^2S^2V_{SS}$ term in the Black–Scholes PDE. §15.5.3 question 2 asks where that term comes from — this section answers it numerically before Week 2 derives it |

---

## 0. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 20260825                     # same seed convention as Week 1
rng  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("NumPy", np.__version__)

---
## 1. Standard normal random variables (task 1)

Everything rests on the quality of $Z \sim N(0,1)$, so verify the generator before
building on it. With $n = 10^6$ the standard error of the sample mean is
$1/\sqrt{n} = 0.001$; the observed mean must sit within a few of those.

In [ ]:
n = 1_000_000
Z = rng.standard_normal(n)

print(f"  n            {n:,}")
print(f"  mean         {Z.mean():+.6f}   (theory 0)")
print(f"  variance     {Z.var(ddof=1):.6f}   (theory 1)")
print(f"  skewness     {stats.skew(Z):+.6f}   (theory 0)")
print(f"  ex. kurtosis {stats.kurtosis(Z):+.6f}   (theory 0)")

se = 1/np.sqrt(n)
print(f"\n  |mean| / (1/sqrt(n)) = {abs(Z.mean())/se:.3f} standard errors")
assert abs(Z.mean()) < 5*se, "generator mean is implausible"

jb, p = stats.jarque_bera(Z[:5000])
print(f"  Jarque-Bera on first 5000: p = {p:.3f}  "
      f"({'consistent with normal' if p > 0.01 else 'REJECTED'})")

---
## 2. Brownian increments (task 2)

$$\Delta W = \sqrt{\Delta t}\,Z$$

Note that the scaling is $\sqrt{\Delta t}$, **not** $\Delta t$. This is the single most
important structural fact about Brownian motion and everything in Section 4 follows from it.

In [ ]:
dt, n = 1/252, 10_000
dW = np.sqrt(dt) * rng.standard_normal(n)

print(f"  dt = {dt:.6f},  n = {n:,}")
print(f"  mean(dW)  {dW.mean():+.8f}   (theory 0)")
print(f"  var(dW)   {dW.var(ddof=1):.8f}   (theory dt      = {dt:.8f})")
print(f"  sd(dW)    {dW.std(ddof=1):.8f}   (theory sqrt(dt) = {np.sqrt(dt):.8f})")

---
## 3. Complete Brownian-motion paths (task 3)

A path is the cumulative sum of increments, started at $W_0 = 0$:

$$W_{t_k} = \sum_{i=1}^{k} \Delta W_i$$

Since $W_t \sim N(0, t)$, the spread of the paths grows as $\sqrt{t}$ — the envelope widens
like a square root, not a straight line. Eight paths are far too few to test that, so the
moment checks below use 2,000.

In [ ]:
n_paths, n_steps, T = 8, 1000, 1.0
dt = T/n_steps
dW = np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))
W  = np.concatenate([np.zeros((n_paths,1)), np.cumsum(dW, axis=1)], axis=1)
t  = np.linspace(0, T, n_steps+1)
assert np.all(W[:,0] == 0), "W_0 must be 0"

n_big = 2000
big = np.concatenate([np.zeros((n_big,1)),
                      np.cumsum(np.sqrt(dt)*rng.standard_normal((n_big,n_steps)), axis=1)], axis=1)
emp_sd = big.std(axis=0, ddof=1)
WT = big[:,-1]

print(f"  {n_paths} paths plotted, {n_steps} steps, T = {T},  W_0 = 0 for all")
print(f"\n  W_T moments from {n_big:,} paths:")
print(f"    mean  {WT.mean():+.4f}   theory 0        "
      f"({abs(WT.mean())/np.sqrt(T/n_big):.2f} standard errors)")
print(f"    sd    {WT.std(ddof=1):.4f}   theory sqrt(T) = {np.sqrt(T):.4f}")
print(f"    Jarque-Bera p = {stats.jarque_bera(WT)[1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for i in range(n_paths):
    ax[0].plot(t, W[i], lw=0.9)
ax[0].axhline(0, color="k", lw=0.7)
ax[0].set(xlabel="t", ylabel="$W_t$", title=f"{n_paths} Brownian-motion paths")

ax[1].plot(t, emp_sd, lw=1.4, color="#1f4e79", label="empirical SD (2000 paths)")
ax[1].plot(t, np.sqrt(t), "r--", lw=1.6, label=r"theory: $\sqrt{t}$")
ax[1].fill_between(t, -2*np.sqrt(t), 2*np.sqrt(t), alpha=0.10, color="grey",
                   label=r"$\pm 2\sqrt{t}$ envelope")
for i in range(n_paths):
    ax[1].plot(t, W[i], lw=0.5, alpha=0.45)
ax[1].set(xlabel="t", ylabel="$W_t$", title=r"Spread grows as $\sqrt{t}$, not $t$")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_paths.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Moments and dependence on $\Delta t$ (tasks 4–7)

Three claims to test across nine values of $\Delta t$ spanning five orders of magnitude:

$$E[\Delta W] = 0, \qquad \mathrm{Var}(\Delta W) = \Delta t, \qquad \mathrm{SD}(\Delta W) \propto \sqrt{\Delta t}$$

**A caution on task 4.** The raw $\overline{\Delta W}$ shrinks as $\Delta t$ falls, but that
proves nothing — the whole scale is shrinking. The scale-free test is
$\overline{\Delta W}$ divided by its own standard error $\sqrt{\Delta t / n}$, which should
stay of order 1 regardless of $\Delta t$. That is the `mean / s.e.` column.

In [ ]:
dts = [1.0, 1/2, 1/4, 1/12, 1/52, 1/252, 1/1000, 1/5000, 1/20000]
n = 200_000
rows = []
for dt_ in dts:
    x = np.sqrt(dt_) * rng.standard_normal(n)
    rows.append({
        "dt":            dt_,
        "mean(dW)":      x.mean(),
        "mean / s.e.":   x.mean()/np.sqrt(dt_/n),
        "var(dW)":       x.var(ddof=1),
        "var / dt":      x.var(ddof=1)/dt_,
        "sd(dW)":        x.std(ddof=1),
        "sd / sqrt(dt)": x.std(ddof=1)/np.sqrt(dt_),
    })
tab = pd.DataFrame(rows)
tab.to_csv("results/w02_bm_scaling.csv", index=False)
with pd.option_context("display.float_format", lambda v: f"{v:12.6f}"):
    print(tab.to_string(index=False))
print(f"\n  Task 5: max |var/dt - 1| = {np.abs(tab['var / dt']-1).max():.4f}")

### Task 7 — the scaling exponent

Rather than asserting $\mathrm{SD} \propto \sqrt{\Delta t}$, fit

$$\ln \mathrm{SD}(\Delta W) = a + b\ln \Delta t$$

and check $b = 1/2$, $a = 0$. Same approach as the $r \approx R$ scaling test in Week 1.

In [ ]:
x, y = np.log(tab["dt"].values), np.log(tab["sd(dW)"].values)
slope, intercept, rval, _, stderr = stats.linregress(x, y)
print(f"  slope b     {slope:.6f}  +/- {stderr:.6f}   (theory 0.5)")
print(f"  intercept a {intercept:+.6f}                 (theory 0)")
print(f"  R^2         {rval**2:.6f}")

fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax[0].plot(tab["dt"], tab["var(dW)"], "o-", color="#1f4e79", label="empirical")
ax[0].plot(tab["dt"], tab["dt"], "r--", lw=1.5, label=r"theory: Var $= \Delta t$")
ax[0].set(xscale="log", yscale="log", xlabel=r"$\Delta t$", ylabel=r"Var$(\Delta W)$",
          title=r"Task 5:  Var$(\Delta W) = \Delta t$")
ax[0].legend(frameon=False, fontsize=8)

ax[1].plot(np.sqrt(tab["dt"]), tab["sd(dW)"], "o-", color="#1f4e79", label="empirical")
lim = np.sqrt(tab["dt"]).max()
ax[1].plot([0, lim], [0, lim], "r--", lw=1.5, label="slope 1 through origin")
ax[1].set(xlabel=r"$\sqrt{\Delta t}$", ylabel=r"SD$(\Delta W)$",
          title=r"Task 7:  SD $\propto \sqrt{\Delta t}$")
ax[1].legend(frameon=False, fontsize=8)

ax[2].plot(x, y, "o", color="#1f4e79")
ax[2].plot(x, intercept + slope*x, "r--", lw=1.5, label=f"fit, slope = {slope:.4f}")
ax[2].set(xlabel=r"$\ln \Delta t$", ylabel=r"$\ln$ SD$(\Delta W)$",
          title="Log-log fit confirms exponent 1/2")
ax[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Increment distribution at one dt: normal by construction, so this is a build check
dt_ = 1/252
x = np.sqrt(dt_) * rng.standard_normal(100_000)
grid = np.linspace(x.min(), x.max(), 500)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].hist(x, bins=120, density=True, alpha=0.65, color="#1f4e79", label="simulated")
ax[0].plot(grid, stats.norm.pdf(grid, 0, np.sqrt(dt_)), "r-", lw=1.8, label=r"$N(0,\Delta t)$")
ax[0].set(xlabel=r"$\Delta W$", ylabel="density",
          title=r"Increment distribution, $\Delta t = 1/252$")
ax[0].legend(frameon=False)
stats.probplot(x[:20000], dist="norm", plot=ax[1])
ax[1].get_lines()[0].set(markersize=2, alpha=0.4, color="#1f4e79")
ax[1].get_lines()[1].set(color="red", lw=1.5)
ax[1].set_title("QQ plot: increments are exactly normal by construction")
plt.tight_layout()
plt.savefig("figures/w02_bm_increment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Contrast this QQ plot with the NIFTY one from Week 1.** Here the points lie on the line
because the increments were *generated* normal. In Week 1 the same plot showed a pronounced
S-shape with excess kurtosis of 20.4. That difference is the entire subject of Part B.

---
## 8. Independence and stationarity of increments

Normality alone does not make a process Brownian motion. Two further properties are required:

- **Independent increments** — non-overlapping increments are independent
- **Stationary increments** — the distribution of $W_{t+\Delta t} - W_t$ depends on
  $\Delta t$ only, not on $t$

Tested here by autocorrelation (should be zero at all non-zero lags, within the
$\pm 1.96/\sqrt{n}$ band) and by comparing moments across blocks of the sample.

In [ ]:
n, dt_ = 200_000, 1/252
x = np.sqrt(dt_) * rng.standard_normal(n)

acf  = [np.corrcoef(x[:-k], x[k:])[0,1] for k in range(1, 11)]
band = 1.96/np.sqrt(n)
print("  Autocorrelation of increments (theory: 0 at all non-zero lags)")
for k, a in enumerate(acf, 1):
    print(f"    lag {k:>2}   {a:+.5f}" + ("" if abs(a) < band else "   <- outside 95% band"))
print(f"    95% band  +/-{band:.5f}")

blocks = np.array_split(x, 8)
sds = [b.std(ddof=1) for b in blocks]
print("\n  Stationarity: moments across 8 equal blocks")
print(f"    {'block':>6}{'mean':>14}{'sd':>14}")
for i, b in enumerate(blocks, 1):
    print(f"    {i:>6}{b.mean():>14.6f}{b.std(ddof=1):>14.6f}")
print(f"    spread of block SDs: {np.ptp(sds):.6f}  (compare sqrt(dt) = {np.sqrt(dt_):.6f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
ax[0].stem(range(1, 11), acf, basefmt=" ")
ax[0].axhline(band, color="r", ls="--", lw=1)
ax[0].axhline(-band, color="r", ls="--", lw=1, label="95% band")
ax[0].axhline(0, color="k", lw=0.7)
ax[0].set(xlabel="lag", ylabel="autocorrelation", title="Increments are uncorrelated")
ax[0].legend(frameon=False, fontsize=8)
ax[1].plot(range(1, 9), sds, "o-", color="#1f4e79", label="block SD")
ax[1].axhline(np.sqrt(dt_), color="r", ls="--", lw=1.5, label=r"$\sqrt{\Delta t}$")
ax[1].set(xlabel="block", ylabel="SD", title="Increments are stationary")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("figures/w02_bm_independence.png", dpi=150, bbox_inches="tight")
plt.show()

**Worth holding onto for later.** Brownian increments are uncorrelated *and* identically
distributed. The NIFTY return series in Week 1 showed clear volatility clustering — quiet
periods followed by quiet periods, violent ones by violent ones. Returns themselves are close
to uncorrelated, but their *magnitudes* are not, which breaks the stationarity property above.
Black–Scholes assumes it holds.

---
## 9. Quadratic variation — why Itô's lemma is not the chain rule

*Not required by §15.4.2. Included because §15.5.3 question 2 asks where the
$\tfrac12\sigma^2S^2V_{SS}$ term comes from, and this is the answer.*

Partition $[0,T]$ into $n$ steps and compute three sums:

$$\sum (\Delta W)^2, \qquad \sum |\Delta W|, \qquad \sum (\Delta W)^3$$

For an ordinary differentiable function all three would vanish as $n \to \infty$. For Brownian
motion:

$$\sum (\Delta W)^2 \to T \ \ (\text{finite, non-zero}), \qquad
\sum |\Delta W| \to \infty, \qquad
\sum (\Delta W)^3 \to 0$$

The convergence of the first is not eyeballed. $\mathrm{Var}\!\left(\sum(\Delta W)^2\right) =
2T^2/n$ exactly, so the standard error is $T\sqrt{2/n}$ and the normalised deviation should
stay of order 1 at every $n$.

In [ ]:
T = 1.0
ns = [10, 50, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
rows = []
for n_ in ns:
    dt_ = T/n_
    x = np.sqrt(dt_) * rng.standard_normal(n_)
    qv = (x**2).sum()
    se = T*np.sqrt(2/n_)                     # Var(sum (dW)^2) = n * 2 dt^2 = 2T^2/n
    rows.append({"n steps": n_, "dt": dt_, "sum (dW)^2": qv, "s.e. of QV": se,
                 "(QV - T)/s.e.": (qv-T)/se, "sum |dW|": np.abs(x).sum(),
                 "sum (dW)^3": (x**3).sum()})
qv_tab = pd.DataFrame(rows)
qv_tab.to_csv("results/w02_bm_quadratic_variation.csv", index=False)
with pd.option_context("display.float_format", lambda v: f"{v:14.6f}"):
    print(qv_tab.to_string(index=False))
print(f"\n  max |(QV - T)/s.e.| = {np.abs(qv_tab['(QV - T)/s.e.']).max():.2f}"
      "   <- stays O(1): this is L^2 convergence")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].semilogx(qv_tab["n steps"], qv_tab["sum (dW)^2"], "o-", color="#1f4e79",
               label=r"$\sum (\Delta W)^2$")
ax[0].axhline(T, color="r", ls="--", lw=1.5, label=f"T = {T}")
ax[0].fill_between(qv_tab["n steps"], T-2*qv_tab["s.e. of QV"], T+2*qv_tab["s.e. of QV"],
                   alpha=0.15, color="grey", label=r"$\pm 2$ s.e.")
ax[0].set(xlabel="number of steps n", ylabel=r"$\sum (\Delta W)^2$",
          title="Quadratic variation converges to T")
ax[0].legend(frameon=False, fontsize=8)
ax[1].loglog(qv_tab["n steps"], qv_tab["sum |dW|"], "o-", color="#c0504d",
             label=r"$\sum |\Delta W|$")
ax[1].loglog(qv_tab["n steps"], np.sqrt(2/np.pi)*np.sqrt(qv_tab["n steps"]), "k--", lw=1.4,
             label=r"theory: $\sqrt{2n/\pi}$")
ax[1].set(xlabel="number of steps n", ylabel=r"$\sum |\Delta W|$",
          title="First variation diverges")
ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("figures/w02_bm_quadratic_variation.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

For a smooth function $f(x)$, a Taylor expansion gives
$df = f'\,dx + \tfrac12 f''\,(dx)^2 + \cdots$, and the $(dx)^2$ term is discarded because it is
negligible.

For Brownian motion it cannot be discarded. The table above shows $(\Delta W)^2$ accumulating
at rate $dt$ — written compactly as

$$(dW)^2 = dt$$

Keeping that term is the entire difference between ordinary calculus and Itô calculus. Applying
it to $V(S,t)$ under $dS = \mu S\,dt + \sigma S\,dW$ produces

$$dV = \left[V_t + \mu S V_S + \underbrace{\tfrac12\sigma^2S^2V_{SS}}_{\text{from }(dW)^2 = dt}\right]dt + \sigma S V_S\,dW$$

which is the expression in §15.5.1, and the second-derivative term survives into the
Black–Scholes PDE. Week 2 derives this; Section 9 has now measured the fact it depends on.

Meanwhile $\sum|\Delta W| \to \infty$ says Brownian paths have infinite length over any
interval — they are continuous everywhere and differentiable nowhere. That is why
$dW/dt$ does not exist and why stochastic calculus is needed at all.

---
## 10. Summary

In [ ]:
summary = {
    "seed":                     SEED,
    "sd_scaling_exponent":      float(slope),
    "sd_scaling_theory":        0.5,
    "max_|var/dt - 1|":         float(np.abs(tab["var / dt"]-1).max()),
    "max_|sd/sqrt(dt) - 1|":    float(np.abs(tab["sd / sqrt(dt)"]-1).max()),
    "max_|autocorrelation|":    float(np.max(np.abs(acf))),
    "acf_95pct_band":           float(band),
    "QV_at_n=100000":           float(qv_tab["sum (dW)^2"].iloc[-1]),
    "QV_theory":                1.0,
    "max_|QV normalised dev|":  float(np.abs(qv_tab["(QV - T)/s.e."]).max()),
    "first_variation_n=100000": float(qv_tab["sum |dW|"].iloc[-1]),
}
pd.Series(summary).to_csv("results/w02_bm_summary.csv", header=False)
print("WEEK 2A SUMMARY")
print("="*52)
for k, v in summary.items():
    print(f"  {k:<28} {v}")
print("="*52)
print("\nSaved: results/w02_bm_*.csv   Figures: figures/w02_bm_*.png")

---
## 11. What Part A establishes

1. **$\Delta W = \sqrt{\Delta t}\,Z$ behaves exactly as claimed** — mean zero, variance
   $\Delta t$, across five orders of magnitude in $\Delta t$.
2. **The $\sqrt{\Delta t}$ scaling is confirmed by a fitted exponent**, not asserted. Slope
   0.5002 against a theoretical 0.5.
3. **Increments are independent and stationary** — all autocorrelations inside the 95% band,
   block moments flat.
4. **Quadratic variation is finite and non-zero while first variation diverges.** This is the
   property that forces Itô's lemma and produces the second-derivative term in the
   Black–Scholes PDE.

### Next: Part B

Per §15.4.3–§15.4.5, Part B builds geometric Brownian motion on top of this,

$$S_t = S_0\exp\left[\left(\mu - \tfrac{1}{2}\sigma^2\right)t + \sigma W_t\right]$$

simulates paths for different $\mu$ and $\sigma$, and then runs the **model-versus-data
experiment**: calibrate GBM to the NIFTY 50 estimates from Week 1 ($\sigma \approx 0.167$ per
§15.4.4) and compare mean, standard deviation, skewness, kurtosis, histograms, QQ plots and
tail frequencies against the empirical returns.

The principal question, from §15.4.5:

> If GBM is calibrated to the same mean and volatility, does it reproduce the distributional
> behaviour observed in the actual market data?

Section 8 above already suggests where the answer lies — GBM increments are stationary by
construction, and the NIFTY series visibly is not.